<a href="https://colab.research.google.com/github/shahzad-jatoi/flyrank-ml-internship-starter/blob/main/work%20/notebooks/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/shahzad-jatoi/flyrank-ml-internship-starter/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [ ]:
#setup cell
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/shahzad-jatoi/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "duckdb"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")

import duckdb, pandas as pd

if IN_COLAB:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
else:
    hf_token = os.environ.get("HF_TOKEN")

con = duckdb.connect()
con.execute(f"CREATE SECRET (TYPE huggingface, TOKEN '{hf_token}')")
WAREHOUSE = "hf://datasets/FlyRank/internship-warehouse"
print("DuckDB ready, warehouse path set.")

DuckDB ready, warehouse path set.


## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row means:** one page's daily performance snapshot (impressions, clicks, position, CTR for that day)
- **Table(s):** `fact_content_daily_performance`, partitioned by `month=YYYY-MM`
- **Time window:** a mid-panel month, `month=2026-03`, to avoid the outcome-window trap in the final sealed month
- **What I'd predict/rank:** a refresh-priority score per page — same proxy label as before (declining trend), scored using this month's slice
- **Deliberately excluded:** any client, domain, URL, or query-identifying columns I only touch anonymized IDs and numeric/categorical performance fields, per DATA_USE.md

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

- **Feature:** avg_impressions, avg_clicks, avg_position, avg_ctr, days_present  all pre-decision, observable before any refresh action is taken
- **Label/proxy:** trend_direction (or next-month trend_pct)  what I'd ultimately want to predict or rank by
- **Context:** content_id, date, month partition  identifiers and time bounds, not fed to the model directly but needed to join/group
- **Excluded, with why:** any client name, domain, URL, or raw query text  excluded per DATA_USE.md; also excluding trend_pct itself as a feature since it's the source the label proxy is derived from (would leak the answer)

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [ ]:
schema = con.sql(f"""
    DESCRIBE SELECT * FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/month=2026-03/*.parquet') LIMIT 1
""").df()
print(schema.to_string())

                 column_name column_type null   key default extra
0                report_date        DATE  YES  None    None  None
1             client_hash_id     VARCHAR  YES  None    None  None
2            content_hash_id     VARCHAR  YES  None    None  None
3             client_has_gsc     BOOLEAN  YES  None    None  None
4             client_has_ga4     BOOLEAN  YES  None    None  None
5         gsc_data_available     BOOLEAN  YES  None    None  None
6         ga4_data_available     BOOLEAN  YES  None    None  None
7            gsc_impressions      BIGINT  YES  None    None  None
8                 gsc_clicks      BIGINT  YES  None    None  None
9           gsc_sum_position      BIGINT  YES  None    None  None
10          gsc_avg_position      DOUBLE  YES  None    None  None
11             ga4_pageviews      BIGINT  YES  None    None  None
12              ga4_sessions      BIGINT  YES  None    None  None
13                 ga4_users      BIGINT  YES  None    None  None
14      ga

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

# Query 1: grain check — is one row really one page-day (per client)?
q1 = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) as n
    FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("Duplicate (client, content, date) rows found:", len(q1), "(should be 0 if grain holds)")

# Query 2: row count and date span for this slice
q2 = con.sql(f"""
    SELECT COUNT(*) as row_count,
           MIN(report_date) as start_date,
           MAX(report_date) as end_date,
           COUNT(DISTINCT content_hash_id) as unique_pages,
           COUNT(DISTINCT client_hash_id) as unique_clients
    FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(q2)

# Query 3: availability — filter with IS TRUE, show survival count
q3 = con.sql(f"""
    SELECT COUNT(*) as total_rows,
           SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) as gsc_available_rows,
           SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) as ga4_available_rows
    FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/month=2026-03/*.parquet')
""").df()
print(q3)


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Duplicate (client, content, date) rows found: 0 (should be 0 if grain holds)
   row_count start_date   end_date  unique_pages  unique_clients
0    9841378 2026-03-01 2026-03-31        331437              55


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows  gsc_available_rows  ga4_available_rows
0     9841378           3611061.0            413966.0


In [ ]:
features_df = con.sql(f"""
    SELECT
        content_hash_id,
        AVG(gsc_impressions) as avg_impressions,          -- 1. known: observed daily search impressions
        AVG(gsc_clicks) as avg_clicks,                     -- 2. known: observed daily clicks
        AVG(gsc_avg_position) as avg_position,             -- 3. known: measured ranking position, pre-decision
        AVG(gsc_clicks * 1.0 / NULLIF(gsc_impressions, 0)) as avg_ctr,  -- 4. known: derived from 1&2, both pre-decision
        COUNT(DISTINCT report_date) as days_present         -- 5. known: just a count of observed days this window
    FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/month=2026-03/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

print(features_df.head())
print(f"\nPages with GSC data in this window: {features_df.shape[0]}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

            content_hash_id  avg_impressions  avg_clicks  avg_position  \
0  content_2e6360ad20fd7107        29.000000    0.032258      5.145765   
1  content_ac8663da7484669a         2.000000    0.000000      4.909314   
2  content_d49a012dcb924e31        10.612903    0.000000      5.177774   
3  content_614baf2af4330bd7        24.903226    0.032258      4.685335   
4  content_4a1ca0fa5c177e0c         1.400000    0.000000      4.266667   

    avg_ctr  days_present  
0  0.001613            31  
1  0.000000            17  
2  0.000000            31  
3  0.000922            31  
4  0.000000            10  

Pages with GSC data in this window: 176738


In [ ]:
# Pull a later month as the "future outcome" to define a proxy label
outcome_df = con.sql(f"""
    SELECT content_hash_id, AVG(gsc_avg_position) as avg_position_next_month
    FROM read_parquet('{WAREHOUSE}/fact_content_daily_performance/month=2026-04/*.parquet')
    WHERE gsc_data_available IS TRUE
    GROUP BY content_hash_id
""").df()

merged = features_df.merge(outcome_df, on="content_hash_id", how="inner")
merged["declined"] = (merged["avg_position_next_month"] > merged["avg_position"]).astype(int)  # position got worse = higher number = declined

# --- LEAK: add the next-month value directly as a "feature" ---
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X_leaky = merged[["avg_impressions", "avg_clicks", "avg_position", "avg_ctr", "avg_position_next_month"]]
y = merged["declined"]
leaky_model = LogisticRegression(max_iter=500).fit(X_leaky, y)
leaky_acc = accuracy_score(y, leaky_model.predict(X_leaky))
print(f"WITH leak (avg_position_next_month included): accuracy = {leaky_acc:.3f}  <- unrealistically high, it's cheating")

# --- HONEST: remove the leaked column ---
X_honest = merged[["avg_impressions", "avg_clicks", "avg_position", "avg_ctr"]]
honest_model = LogisticRegression(max_iter=500).fit(X_honest, y)
honest_acc = accuracy_score(y, honest_model.predict(X_honest))
print(f"WITHOUT leak (only pre-decision features): accuracy = {honest_acc:.3f}  <- the real, defensible number")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

WITH leak (avg_position_next_month included): accuracy = 0.999  <- unrealistically high, it's cheating
WITHOUT leak (only pre-decision features): accuracy = 0.690  <- the real, defensible number


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

his slice is an unbalanced panel by design `client_has_gsc` and `client_has_ga4` (and per-row `gsc_data_available` / `ga4_data_available`) show that not every client-page-day has both data sources present, so filtering on `gsc_data_available IS TRUE` already drops an unknown share of rows before any modeling starts; this data can't tell me what happened on days or for clients where GSC simply wasn't syncing.

Per-client history also isn't uniform some clients only have data starting partway through the warehouse's timeline, so `month=2026-03` doesn't represent every client equally; a client that joined FlyRank later may be thin or entirely absent in this window, and I have no way to distinguish "no data because nothing happened" from "no data because we weren't tracking them yet."

Finally, this is anonymized, aggregated daily performance hash keys mean I can measure patterns across pseudonymous pages and clients, but I can never tie a result back to what the page actually is, what it's about, or why a human made a specific editorial choice on it. Any explanation for *why* a pattern holds is inference from these numbers, not something the data itself confirms.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.